# 03: Risk Analysis & Probability Metrics

Complete guide to risk metrics and probability calculations in Argo.

## What You'll Learn

- **Value at Risk (VaR):** Quantify potential losses
- **Conditional VaR (CVaR):** Expected loss beyond VaR
- **Probability Metrics:** Exceeding, below, between, target achievement
- **Financial Risk:** Portfolio analysis and decision making
- **Project Risk:** Timeline and budget risk assessment

## Prerequisites

- Argo package built (`npm run build --workspace=@argo/core`)
- TypeScript kernel (tslab) installed
- Familiarity with distributions (Notebook 01) and statistics (Notebook 02)

## Setup and Imports

In [ ]:
// Import Argo risk analysis functions
const argo = require('../packages/argo-core/dist/index');

const {
  // Distributions (for generating scenarios)
  NormalDistribution,
  TriangularDistribution,
  LogNormalDistribution,
  SimpleRNG,
  
  // Statistical functions
  mean,
  median,
  standardDeviation,
  percentile,
  
  // Risk Metrics (6 functions)
  valueAtRisk,
  conditionalVaR,
  probabilityExceeding,
  probabilityBelow,
  probabilityBetween,
  probabilityOfTarget
} = argo;

console.log('✅ Risk analysis functions loaded successfully!');
console.log('Risk metrics available: 6 functions');

## Generate Risk Scenarios

Let's create realistic scenarios for risk analysis.

In [ ]:
// Scenario 1: Portfolio Returns (normally distributed)
const rng1 = new SimpleRNG(42);
const returnsDist = new NormalDistribution(0.08, 0.15); // 8% mean, 15% volatility
const portfolioReturns = [];
for (let i = 0; i < 10000; i++) {
  portfolioReturns.push(returnsDist.sample(rng1));
}

// Scenario 2: Project Costs (right-skewed)
const rng2 = new SimpleRNG(123);
const costDist = new TriangularDistribution(800000, 1000000, 1500000);
const projectCosts = [];
for (let i = 0; i < 10000; i++) {
  projectCosts.push(costDist.sample(rng2));
}

// Scenario 3: Revenue (log-normal)
const rng3 = new SimpleRNG(456);
const revenueDist = new LogNormalDistribution(13.8, 0.3);
const revenues = [];
for (let i = 0; i < 10000; i++) {
  revenues.push(revenueDist.sample(rng3));
}

console.log('Risk scenarios generated:');
console.log(`  Portfolio returns: ${portfolioReturns.length} scenarios`);
console.log(`    Mean: ${(mean(portfolioReturns)*100).toFixed(2)}%`);
console.log(`    Std Dev: ${(standardDeviation(portfolioReturns)*100).toFixed(2)}%`);
console.log(`  Project costs: ${projectCosts.length} scenarios`);
console.log(`    Mean: $${(mean(projectCosts)/1000).toFixed(0)}k`);
console.log(`  Revenues: ${revenues.length} scenarios`);
console.log(`    Mean: $${(mean(revenues)/1000000).toFixed(2)}M`);

---

# Part 1: Value at Risk (VaR)

**VaR answers:** "What's the maximum loss I might face with X% confidence?"

**Example:** "There's a 95% chance my losses won't exceed $50,000."

**Use Cases:**
- Portfolio risk management
- Capital requirements (banking)
- Risk budgeting

In [ ]:
console.log('=== Value at Risk (VaR) ===\n');

// Investment portfolio example
const portfolioValue = 1000000; // $1M portfolio

// Calculate VaR at different confidence levels
const var90 = valueAtRisk(portfolioReturns, 0.90);
const var95 = valueAtRisk(portfolioReturns, 0.95);
const var99 = valueAtRisk(portfolioReturns, 0.99);

console.log('Portfolio Value at Risk:');
console.log(`  Portfolio value: $${(portfolioValue/1000).toFixed(0)}k\n`);

console.log(`  VaR (90% confidence): ${(var90*100).toFixed(2)}%`);
console.log(`    = $${(Math.abs(var90) * portfolioValue / 1000).toFixed(0)}k maximum loss`);
console.log(`    Interpretation: 90% chance loss won't exceed this\n`);

console.log(`  VaR (95% confidence): ${(var95*100).toFixed(2)}%`);
console.log(`    = $${(Math.abs(var95) * portfolioValue / 1000).toFixed(0)}k maximum loss`);
console.log(`    Most common standard (Basel II/III)\n`);

console.log(`  VaR (99% confidence): ${(var99*100).toFixed(2)}%`);
console.log(`    = $${(Math.abs(var99) * portfolioValue / 1000).toFixed(0)}k maximum loss`);
console.log(`    Conservative risk measure\n`);

console.log('Key Points:');
console.log('  - Higher confidence → larger VaR');
console.log('  - VaR is a percentile of the loss distribution');
console.log(`  - 95% VaR = ${percentile(portfolioReturns, 5).toFixed(4)} (5th percentile)`);

---

# Part 2: Conditional Value at Risk (CVaR)

**CVaR answers:** "If losses exceed VaR, what's the expected loss?"

**Also known as:** Expected Shortfall (ES), Average VaR

**Why CVaR > VaR:**
- VaR tells you the threshold, CVaR tells you the expected loss beyond it
- CVaR is "coherent" (mathematically better behaved)
- Captures tail risk better

In [ ]:
console.log('\n=== Conditional Value at Risk (CVaR) ===\n');

// Calculate CVaR at same confidence levels
const cvar90 = conditionalVaR(portfolioReturns, 0.90);
const cvar95 = conditionalVaR(portfolioReturns, 0.95);
const cvar99 = conditionalVaR(portfolioReturns, 0.99);

console.log('Portfolio CVaR Analysis:\n');

console.log(`  90% Confidence:`);
console.log(`    VaR:  ${(var90*100).toFixed(2)}% = $${(Math.abs(var90) * portfolioValue / 1000).toFixed(0)}k`);
console.log(`    CVaR: ${(cvar90*100).toFixed(2)}% = $${(Math.abs(cvar90) * portfolioValue / 1000).toFixed(0)}k`);
console.log(`    If in worst 10%, expect to lose $${(Math.abs(cvar90) * portfolioValue / 1000).toFixed(0)}k on average\n`);

console.log(`  95% Confidence:`);
console.log(`    VaR:  ${(var95*100).toFixed(2)}% = $${(Math.abs(var95) * portfolioValue / 1000).toFixed(0)}k`);
console.log(`    CVaR: ${(cvar95*100).toFixed(2)}% = $${(Math.abs(cvar95) * portfolioValue / 1000).toFixed(0)}k`);
console.log(`    If in worst 5%, expect to lose $${(Math.abs(cvar95) * portfolioValue / 1000).toFixed(0)}k on average\n`);

console.log(`  99% Confidence:`);
console.log(`    VaR:  ${(var99*100).toFixed(2)}% = $${(Math.abs(var99) * portfolioValue / 1000).toFixed(0)}k`);
console.log(`    CVaR: ${(cvar99*100).toFixed(2)}% = $${(Math.abs(cvar99) * portfolioValue / 1000).toFixed(0)}k`);
console.log(`    If in worst 1%, expect to lose $${(Math.abs(cvar99) * portfolioValue / 1000).toFixed(0)}k on average\n`);

console.log('CVaR is always >= VaR (more conservative)');
console.log(`Ratio (95%): CVaR/VaR = ${Math.abs(cvar95/var95).toFixed(2)}x`);

---

# Part 3: Probability Metrics

Answer questions about likelihood of outcomes.

## 3.1 Probability Exceeding

**Answers:** "What's the probability of exceeding a threshold?"

In [ ]:
console.log('\n=== Probability Exceeding ===\n');

// Project cost overrun analysis
const budgetTarget = 1000000;
const costOverrun10 = budgetTarget * 1.10; // 10% over
const costOverrun20 = budgetTarget * 1.20; // 20% over

const pExceed0 = probabilityExceeding(projectCosts, budgetTarget);
const pExceed10 = probabilityExceeding(projectCosts, costOverrun10);
const pExceed20 = probabilityExceeding(projectCosts, costOverrun20);

console.log('Project Cost Overrun Risk:');
console.log(`  Budget: $${(budgetTarget/1000).toFixed(0)}k\n`);

console.log(`  P(Cost > $${(budgetTarget/1000).toFixed(0)}k) = ${(pExceed0*100).toFixed(1)}%`);
console.log(`    → ${(pExceed0*100).toFixed(1)}% chance of any cost overrun\n`);

console.log(`  P(Cost > $${(costOverrun10/1000).toFixed(0)}k) = ${(pExceed10*100).toFixed(1)}%`);
console.log(`    → ${(pExceed10*100).toFixed(1)}% chance of >10% overrun\n`);

console.log(`  P(Cost > $${(costOverrun20/1000).toFixed(0)}k) = ${(pExceed20*100).toFixed(1)}%`);
console.log(`    → ${(pExceed20*100).toFixed(1)}% chance of >20% overrun\n`);

console.log('Use Cases:');
console.log('  - Budget overrun risk');
console.log('  - Performance targets (upside)');
console.log('  - Capacity planning');

## 3.2 Probability Below

**Answers:** "What's the probability of being below a threshold?"

In [ ]:
console.log('\n=== Probability Below ===\n');

// Revenue targets
const revenueTarget = 1000000;
const revenueStretch = 1500000;

const pBelow1M = probabilityBelow(revenues, revenueTarget);
const pBelow15M = probabilityBelow(revenues, revenueStretch);

console.log('Revenue Achievement Analysis:');
console.log(`  Targets: $${(revenueTarget/1000).toFixed(0)}k (base), $${(revenueStretch/1000).toFixed(0)}k (stretch)\n`);

console.log(`  P(Revenue < $${(revenueTarget/1000).toFixed(0)}k) = ${(pBelow1M*100).toFixed(1)}%`);
console.log(`    → ${((1-pBelow1M)*100).toFixed(1)}% chance of meeting base target\n`);

console.log(`  P(Revenue < $${(revenueStretch/1000).toFixed(0)}k) = ${(pBelow15M*100).toFixed(1)}%`);
console.log(`    → ${((1-pBelow15M)*100).toFixed(1)}% chance of meeting stretch target\n`);

console.log('Use Cases:');
console.log('  - Sales target achievement');
console.log('  - Quality standards (defect rates)');
console.log('  - SLA compliance');

## 3.3 Probability Between

**Answers:** "What's the probability of being within a range?"

In [ ]:
console.log('\n=== Probability Between ===\n');

// Portfolio return ranges
const targetReturn = 0.08; // 8% target
const tolerance = 0.05; // ±5%

const pNearTarget = probabilityBetween(portfolioReturns, targetReturn - tolerance, targetReturn + tolerance);
const pPositive = probabilityBetween(portfolioReturns, 0, Infinity);
const pModerate = probabilityBetween(portfolioReturns, -0.10, 0.20);

console.log('Portfolio Return Ranges:\n');

console.log(`  P(${(targetReturn-tolerance)*100}% < Return < ${(targetReturn+tolerance)*100}%) = ${(pNearTarget*100).toFixed(1)}%`);
console.log(`    → ${(pNearTarget*100).toFixed(1)}% chance within ±5% of target\n`);

console.log(`  P(Return > 0%) = ${(pPositive*100).toFixed(1)}%`);
console.log(`    → ${(pPositive*100).toFixed(1)}% chance of positive returns\n`);

console.log(`  P(-10% < Return < 20%) = ${(pModerate*100).toFixed(1)}%`);
console.log(`    → ${(pModerate*100).toFixed(1)}% chance of moderate outcomes\n`);

console.log('Use Cases:');
console.log('  - Acceptable outcome ranges');
console.log('  - Quality control (specification limits)');
console.log('  - Moderate scenario planning');

## 3.4 Probability of Target Achievement

**Answers:** "What's the probability of hitting a target within tolerance?"

In [ ]:
console.log('\n=== Probability of Target ===\n');

// Cost target with acceptable variance
const costTarget = 1000000;
const costTolerance = 50000; // ±$50k acceptable

const pCostTarget = probabilityOfTarget(projectCosts, costTarget, costTolerance);

console.log('Project Cost Target:');
console.log(`  Target: $${(costTarget/1000).toFixed(0)}k`);
console.log(`  Tolerance: ±$${(costTolerance/1000).toFixed(0)}k`);
console.log(`  Acceptable range: [$${((costTarget-costTolerance)/1000).toFixed(0)}k, $${((costTarget+costTolerance)/1000).toFixed(0)}k]\n`);

console.log(`  P(|Cost - Target| ≤ Tolerance) = ${(pCostTarget*100).toFixed(1)}%`);
console.log(`    → ${(pCostTarget*100).toFixed(1)}% chance of hitting target\n`);

// Revenue target
const revTarget = 1200000;
const revTolerance = 100000;
const pRevTarget = probabilityOfTarget(revenues, revTarget, revTolerance);

console.log('Revenue Target:');
console.log(`  Target: $${(revTarget/1000).toFixed(0)}k ± $${(revTolerance/1000).toFixed(0)}k`);
console.log(`  P(hitting target) = ${(pRevTarget*100).toFixed(1)}%\n`);

console.log('Use Cases:');
console.log('  - Goal achievement probability');
console.log('  - Precision manufacturing');
console.log('  - Project delivery on target');

---

# Part 4: Real-World Example

## Investment Portfolio Risk Report

In [ ]:
console.log('\n========================================');
console.log('   PORTFOLIO RISK ANALYSIS REPORT');
console.log('========================================\n');

const portfolio = 1000000;

// 1. Summary Statistics
console.log('1. Portfolio Summary:');
console.log(`   Value: $${(portfolio/1000).toFixed(0)}k`);
console.log(`   Expected Return: ${(mean(portfolioReturns)*100).toFixed(2)}% annually`);
console.log(`   Volatility: ${(standardDeviation(portfolioReturns)*100).toFixed(2)}%`);
console.log(`   Median Return: ${(median(portfolioReturns)*100).toFixed(2)}%\n`);

// 2. Risk Metrics
console.log('2. Risk Metrics (95% confidence):');
const var95_dollar = Math.abs(var95) * portfolio;
const cvar95_dollar = Math.abs(cvar95) * portfolio;
console.log(`   VaR (95%): $${(var95_dollar/1000).toFixed(0)}k`);
console.log(`   CVaR (95%): $${(cvar95_dollar/1000).toFixed(0)}k`);
console.log(`   Max expected loss: ${(Math.abs(var95)*100).toFixed(2)}% of portfolio\n`);

// 3. Probability Analysis
console.log('3. Outcome Probabilities:');
const pLoss = probabilityBelow(portfolioReturns, 0);
const pTarget = probabilityBetween(portfolioReturns, 0.05, 0.15);
const pExcellent = probabilityExceeding(portfolioReturns, 0.20);
console.log(`   P(Loss): ${(pLoss*100).toFixed(1)}%`);
console.log(`   P(5-15% return): ${(pTarget*100).toFixed(1)}%`);
console.log(`   P(>20% return): ${(pExcellent*100).toFixed(1)}%\n`);

// 4. Scenario Analysis
console.log('4. Scenario Analysis:');
const bearMarket = -0.20; // -20%
const bullMarket = 0.30;  // +30%
console.log(`   P(Bear Market < -20%): ${(probabilityBelow(portfolioReturns, bearMarket)*100).toFixed(1)}%`);
console.log(`   P(Bull Market > 30%): ${(probabilityExceeding(portfolioReturns, bullMarket)*100).toFixed(1)}%\n`);

// 5. Risk Rating
console.log('5. Risk Rating:');
const sharpeRatio = mean(portfolioReturns) / standardDeviation(portfolioReturns);
console.log(`   Sharpe Ratio: ${sharpeRatio.toFixed(2)}`);
if (sharpeRatio > 1) {
  console.log(`   Assessment: Good risk-adjusted returns`);
} else if (sharpeRatio > 0.5) {
  console.log(`   Assessment: Acceptable risk-adjusted returns`);
} else {
  console.log(`   Assessment: Poor risk-adjusted returns`);
}

console.log('\n========================================');
console.log('         END OF RISK REPORT');
console.log('========================================');

---

## Summary

### ✅ What We Covered

**6 Risk Metric Functions:**

1. **valueAtRisk(data, confidence)** - Maximum loss at confidence level
   - VaR(95%) = "95% sure loss won't exceed this"
   - Used for capital requirements, risk limits

2. **conditionalVaR(data, confidence)** - Expected loss beyond VaR
   - CVaR(95%) = "Average loss in worst 5% of cases"
   - More conservative, captures tail risk

3. **probabilityExceeding(data, threshold)** - P(X > threshold)
   - "What's the chance of exceeding this value?"
   - Overrun risk, upside opportunities

4. **probabilityBelow(data, threshold)** - P(X < threshold)
   - "What's the chance of being below this value?"
   - Target achievement, downside risk

5. **probabilityBetween(data, lower, upper)** - P(lower ≤ X ≤ upper)
   - "What's the chance of being in this range?"
   - Acceptable outcomes, moderate scenarios

6. **probabilityOfTarget(data, target, tolerance)** - P(|X - target| ≤ tolerance)
   - "What's the chance of hitting the target?"
   - Goal achievement, precision metrics

### 🎯 Key Concepts

- **VaR vs CVaR:** VaR shows threshold, CVaR shows expected loss beyond it
- **Confidence Levels:** 90%, 95%, 99% are standard (higher = more conservative)
- **Tail Risk:** CVaR captures extreme losses better than VaR
- **Probability Metrics:** Answer "what if" questions about outcomes
- **Risk Reports:** Combine multiple metrics for comprehensive analysis

### 📚 Applications

- **Finance:** Portfolio risk, capital allocation, stress testing
- **Projects:** Cost overruns, schedule delays, target achievement
- **Operations:** Quality control, capacity planning, SLA compliance
- **Strategy:** Scenario planning, decision analysis, risk budgeting

### 📚 Next Steps

- **Notebook 04:** Monte Carlo Simulation Engine
- **Notebook 05:** CLI Usage and Configuration